# NB01 — Corpus Preparation
**MSK | Goel Lab** · LLM Systematic Review · Method Comparison

**Purpose**: Ingest source documents (PDFs + PubMed abstracts), deidentify, chunk, embed, and build the shared vector store used by Methods 2 and 3.

**Outputs**
- `data/corpus/corpus_metadata.csv` — title, abstract, doi, year, source_db per document
- `data/features/chunk_embeddings.npy` + `data/features/chunk_index.csv`
- Persisted ChromaDB collection at `chroma_db/`

**Dependencies**: `src/methods/standard_rag/langchain_pipeline.py`, `src/utils/io.py`

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.io import load_env, data_dir, project_root
load_env()

from pathlib import Path
import pandas as pd
import numpy as np

## 1. Load documents from PubMed

In [ ]:
from src.methods.standard_rag.langchain_pipeline import load_from_pubmed, load_from_pdfs, load_from_json

PUBMED_QUERY = (
    '("large language model"[tiab] OR "LLM"[tiab] OR "GPT"[tiab]) '
    'AND ("feature extraction"[tiab] OR "information extraction"[tiab]) '
    'AND ("radiology report"[tiab] OR "pathology report"[tiab]) '
    'AND ("2018/01/01"[PDAT] : "2026/12/31"[PDAT])'
)

docs = load_from_pubmed(PUBMED_QUERY, max_docs=500)
print(f'Loaded {len(docs)} documents')

## 2. Chunk documents

In [ ]:
from src.methods.standard_rag.langchain_pipeline import chunk_documents

chunks = chunk_documents(docs)
print(f'{len(docs)} docs → {len(chunks)} chunks')

## 3. Build vector store (run once)

In [ ]:
from src.methods.standard_rag.langchain_pipeline import build_vector_store

vs = build_vector_store(chunks)
print('Vector store built.')

## 4. Save corpus metadata to `data/corpus/`

In [ ]:
rows = [
    {
        'title':     d.metadata.get('title', ''),
        'year':      d.metadata.get('year', ''),
        'source_db': d.metadata.get('source', 'pubmed'),
    }
    for d in docs
]
corpus_df = pd.DataFrame(rows).drop_duplicates(subset=['title'])
out_path = data_dir() / 'corpus' / 'corpus_metadata.csv'
out_path.parent.mkdir(parents=True, exist_ok=True)
corpus_df.to_csv(out_path, index=False)
print(f'Saved {len(corpus_df)} records to {out_path}')